# 07 — Production Streaming: Examples

> Build a FastAPI SSE endpoint and a `ProductionStreamer` class with error recovery.

**What you'll build:**
1. Async SSE generator
2. FastAPI streaming endpoint
3. `ProductionStreamer` class with keepalive, error handling, metadata
4. Python SSE client test

In [ ]:
# !pip install fastapi uvicorn openai httpx python-dotenv rich

In [ ]:
import os, json, asyncio, time
from openai import AsyncOpenAI, APIError
from dotenv import load_dotenv
from rich import print as rprint
from rich.console import Console

load_dotenv()
client = AsyncOpenAI()
console = Console()
print('✅ Setup complete')

---
## Part 1: SSE Generator

In [ ]:
def sse_event(data: dict | str) -> str:
    """Format a dictionary as a Server-Sent Event string."""
    payload = json.dumps(data) if isinstance(data, dict) else data
    return f"data: {payload}\n\n"


async def stream_generator(prompt: str):
    """
    Async generator that yields SSE-formatted events.
    Suitable for use as a FastAPI StreamingResponse generator.
    """
    start = time.perf_counter()
    token_count = 0

    try:
        stream = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            stream=True, max_tokens=200
        )

        async for chunk in stream:
            content = chunk.choices[0].delta.content or ""
            finish = chunk.choices[0].finish_reason

            if content:
                token_count += 1
                yield sse_event({"event": "token", "content": content, "n": token_count})

            if finish:
                elapsed = round((time.perf_counter() - start) * 1000, 1)
                yield sse_event({"event": "done", "tokens": token_count, "latency_ms": elapsed, "finish_reason": finish})
                return

    except APIError as e:
        yield sse_event({"event": "error", "message": str(e)})
    except Exception as e:
        yield sse_event({"event": "error", "message": "Internal error"})


# ── Demo: collect and display SSE events ─────────────────────────────────
print("📡 Simulating SSE event stream:\n")

events = []
async for raw_event in stream_generator("Explain what Server-Sent Events are in 2 sentences."):
    events.append(raw_event)
    # Parse and display
    data_str = raw_event.replace("data: ", "").strip()
    try:
        data = json.loads(data_str)
        if data['event'] == 'token':
            print(data['content'], end="", flush=True)
        elif data['event'] == 'done':
            rprint(f"\n\n[bold green]✅ Done[/bold green] — {data['tokens']} tokens in {data['latency_ms']}ms")
    except:
        pass

---
## Part 2: FastAPI App Definition

In [ ]:
# This cell defines the FastAPI app — run via uvicorn, not in Jupyter

FASTAPI_APP = '''
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from openai import AsyncOpenAI, APIError
import json, asyncio, time, logging

logger = logging.getLogger(__name__)
app = FastAPI(title="LLM Streaming API")
client = AsyncOpenAI()

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["GET"])


def sse(data: dict | str) -> str:
    payload = json.dumps(data) if isinstance(data, dict) else data
    return f"data: {payload}\\n\\n"


async def generate(prompt: str, request: Request):
    start = time.time()
    count = 0
    try:
        stream = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            stream=True, timeout=30.0
        )
        async for chunk in stream:
            if await request.is_disconnected(): return
            c = chunk.choices[0].delta.content or ""
            if c:
                count += 1
                yield sse({"event": "token", "content": c})
            if chunk.choices[0].finish_reason:
                yield sse({"event": "done", "tokens": count,
                           "latency_ms": round((time.time()-start)*1000, 1)})
                return
    except APIError as e:
        yield sse({"event": "error", "message": str(e), "code": e.status_code})
    except Exception:
        yield sse({"event": "error", "message": "Internal error"})


@app.get("/stream")
async def stream_endpoint(prompt: str = "Tell me a joke", request: Request = None):
    return StreamingResponse(
        generate(prompt, request),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive", "X-Accel-Buffering": "no"}
    )


# Run with: uvicorn main:app --reload --port 8000
'''

# Write app to file
with open("/tmp/streaming_app.py", "w") as f:
    f.write(FASTAPI_APP)

print("✅ FastAPI app written to /tmp/streaming_app.py")
print("   Run with: uvicorn streaming_app:app --reload --port 8000")

---
## Part 3: Python SSE Client

In [ ]:
# ── Python SSE client using httpx ──────────────────────────────────────────
# (Uncomment and run after starting the FastAPI server)

# import httpx
#
# def consume_sse(url: str, prompt: str):
#     """Consume an SSE stream from the FastAPI endpoint."""
#     params = {"prompt": prompt}
#     with httpx.Client(timeout=60) as client:
#         with client.stream("GET", url, params=params) as response:
#             for line in response.iter_lines():
#                 if not line or not line.startswith("data: "):
#                     continue
#                 data_str = line[6:]
#                 try:
#                     data = json.loads(data_str)
#                     if data['event'] == 'token':
#                         print(data['content'], end="", flush=True)
#                     elif data['event'] == 'done':
#                         print(f"\n✅ Done: {data['tokens']} tokens in {data['latency_ms']}ms")
#                         break
#                     elif data['event'] == 'error':
#                         print(f"\n❌ Error: {data['message']}")
#                         break
#                 except json.JSONDecodeError:
#                     pass
#
# consume_sse("http://localhost:8000/stream", "Explain streaming in 2 sentences.")

print("ℹ️  Start the FastAPI server first, then uncomment the client above.")

---
## 🎉 Module Complete!

**`07_streaming_responses` is fully built:**

| Topic | Key Skill |
|---|---|
| `01_streaming_basics` | `stream=True`, delta.content, TTFT, throughput |
| `02_openai_streaming` | Context manager, tool call detection, finish_reason |
| `03_streaming_across_providers` | Anthropic, Gemini, LiteLLM unified |
| `04_streaming_in_agent_loops` | Stream reasoning + tool execution in agent loop |
| `05_streaming_tool_calls` | JSON argument accumulation across chunks |
| `06_async_streaming` | asyncio.gather for parallel streams |
| `07_production_streaming` | FastAPI SSE endpoint for production |

**Module 02 — LLM Core Skills: COMPLETE ✅**